# Diabetes Big Data Pipeline:  Apache Spark & MongoDB

### Group 11: Prayusha Poudel, Tuo Yan , Dan Le 

## Section 1: Setup & Imports

In [ ]:
# Install required packages
# Run this cell once, then restart the kernel before continuing
%pip install pyspark pymongo pandas matplotlib seaborn

In [ ]:
import os
import time
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, desc, rank, when
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

print("All imports successful!")

## Section 2: Initialize SparkSession (with MongoDB Connector)


In [ ]:
# Create SparkSession with MongoDB connector
spark = SparkSession.builder \
    .appName("DiabetesBigDataPipeline") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017/diabetes_db.patients") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017/diabetes_db.patients") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession created successfully!")
print(f"Spark version: {spark.version}")

## Section 3: Ingest CSV Data into MongoDB

In [ ]:
# Connect to local MongoDB
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["diabetes_db"]
collection = db["patients"]

# Load CSV with pandas
df_csv = pd.read_csv("diabetes_dataset.csv")

print(f"CSV loaded: {len(df_csv)} rows, {len(df_csv.columns)} columns")
print("Columns:", list(df_csv.columns))
df_csv.head()

In [ ]:
# Insert into MongoDB
# Clear old data first to avoid duplicates on re-runs
collection.delete_many({})

# Convert to list of dicts and insert
records = df_csv.to_dict(orient="records")
collection.insert_many(records)

print(f"Successfully ingested {len(records)} records into MongoDB.")
print(f"Verification — documents in collection: {collection.count_documents({})}")